# MCP Architecture, Protocol Eras, and Trust Boundaries

This guided lab uses the official MCP Python SDK in memory: no network, credentials, or external service. You will inspect a real negotiated session, prove that discovery is not authorization, inject boundary failures, and compare modern MCP with the legacy initialization flow.

**Completion evidence:** negotiated identity/version/capabilities, one allowed tenant read, a discovered-but-unexecuted broad tool, three negative policy cases, a diagnosed legacy trace, and an evidence-coverage score.

## 1. Current protocol mental model

Modern MCP (2026-07-28+) is stateless at the base protocol layer. Requests carry protocol version and client capabilities in per-request metadata; a client can use `server/discover` or recover from an unsupported-version error. Legacy revisions (2025-11-25 and earlier) use `initialize` followed by `notifications/initialized`.

Both eras answer interoperability questions. Neither grants a user permission to invoke a tool or read a resource.

In [ ]:
import runpy
from dataclasses import asdict
from pathlib import Path

module = runpy.run_path(Path("lab.py"))
evidence, payload = await module["run_scenario"]()
asdict(evidence)

## 2. Inspect the real SDK session

The SDK negotiated the version and exposed typed server metadata. Check the surfaces separately: tools are operations, resources are URI-addressed context, and prompts are templates. All descriptions and returned content remain untrusted input.

In [ ]:
print("protocol:", evidence.protocol_version)
print("server:", evidence.server_name, evidence.server_version)
print("capabilities:", evidence.capabilities)
print("tools:", evidence.tools)
print("resources:", evidence.resources)
print("prompts:", evidence.prompts)
print("ticket result:", payload["ticket"])

assert evidence.protocol_version == "2026-07-28"
assert {"ticket.read", "fetch_url"}.issubset(evidence.tools)
assert payload["ticket"]["ok"] is True

## 3. Discovery is not authorization

`fetch_url` is a valid advertised tool, but the trusted host has not approved it. Its argument points to a common cloud-metadata target. The decision must be deterministic application code, not a prompt asking the model to be careful.

In [ ]:
policy = module["HostPolicy"]()
cases = [
    ("broad discovered tool", "fetch_url", {"url": "http://169.254.169.254/"}),
    ("extra argument", "ticket.read", {"ticket_id": "acme-7", "debug": True}),
    ("cross-tenant identifier", "ticket.read", {"ticket_id": "other-7"}),
]

for label, name, arguments in cases:
    decision = policy.authorize_tool("analyst-42", name, arguments)
    print(f"{label:24} allowed={decision.allowed!s:5} reason={decision.reason}")
    assert not decision.allowed

resource_decision = policy.authorize_resource("analyst-42", "support://other/policy")
print("cross-tenant resource    ", asdict(resource_decision))
assert not resource_decision.allowed

The lab records a server-side execution counter. An allow decision causes exactly one `ticket.read`; the denied broad tool remains at zero. This distinguishes *a policy function returned deny* from the stronger claim *denial happened before execution*.

In [ ]:
calls = module["SERVER_CALLS"]
print(calls)
assert calls["ticket.read"] == 1
assert calls["fetch_url"] == 0

## 4. Compare a legacy lifecycle trace

The current session above did not require the legacy initialization handshake. Older and dual-era deployments still make initialization traces operationally relevant, so review them under the rules for their negotiated revision.

In [ ]:
legacy = module["build_legacy_trace"]()
review_legacy = module["review_legacy_trace"]
print("baseline findings:", review_legacy(legacy))

missing_confirmation = [
    event for event in legacy
    if event.get("method") != "notifications/initialized"
]
print("mutated findings:", review_legacy(missing_confirmation))

assert review_legacy(legacy) == []
assert review_legacy(missing_confirmation) == ["legacy client did not confirm initialization"]

## 5. Map the boundaries

For every hop, name the principal, transferred data, permitted authority, retained evidence, and revocation owner. Moving from in-memory to stdio adds a process/filesystem/environment boundary. Moving to Streamable HTTP adds network origin, TLS, authentication, redirect, and egress boundaries.

In [ ]:
for boundary in module["BOUNDARIES"]:
    item = asdict(boundary)
    print(f"{item['name']}: authority={item['authority']}; evidence={item['evidence']}; revoke={item['revocation']}")

## 6. Evaluate the evidence, not the narrative

A reviewable event should identify the protocol/server, capability surface, authenticated subject, action/resource, policy decision, trace correlation, and revocation owner without recording secrets. Score only what the executable evidence actually demonstrates.

In [ ]:
checks = {
    "protocol version": bool(evidence.protocol_version),
    "server identity": bool(evidence.server_name and evidence.server_version),
    "capability inventory": bool(evidence.tools and evidence.resources and evidence.prompts),
    "trace correlation": evidence.trace_id.startswith("trace-"),
    "subject/action/resource": all(d.subject and d.action and d.resource for d in evidence.decisions),
    "allow and deny decisions": {d.allowed for d in evidence.decisions} == {True, False},
    "denied before execution": calls["fetch_url"] == 0,
}
score = sum(checks.values()) / len(checks)
print({name: "PASS" if passed else "FAIL" for name, passed in checks.items()})
print(f"evidence coverage: {score:.0%}")
assert score == 1.0

## 7. Professional inspection workflow

The official MCP Inspector v2 provides Web, TUI, and CLI clients. Against an explicitly reviewed server command, `tools/list --strict` can surface schema portability issues; it does not authorize the tools. For an HTTP deployment, select Streamable HTTP explicitly and protect credentials from shell history and copied diagnostics.

```text
npx @modelcontextprotocol/inspector --cli <server-command> --method tools/list --strict
npx @modelcontextprotocol/inspector --cli https://reviewed.example/mcp --transport http --method tools/list --strict
```

Use the Inspector for exploration and reproducible diagnostics, then encode lasting security expectations in automated contract and adversarial tests.

## 8. Exercises and production transfer

1. Add `ticket.list_open` with no arguments and prove extra arguments are rejected before execution.
2. Retrieve the summary prompt and mutate its text. Prove it cannot alter the host allow-list.
3. Draft a stdio launch policy: pinned executable, arguments, allow-listed environment, filesystem/process isolation, deadline, and shutdown behavior.
4. Draft a Streamable HTTP policy: server/TLS identity, OAuth audience, same-origin redirect behavior, egress, timeouts, result-size limit, evidence, and revocation.
5. Extend the evidence record with latency and a redacted result class; do not add customer content or tokens.

**Reflection:** Why must the server repeat tenant validation after the host approves the call? Which principal and resource server can make the final authorization decision?